In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## Importing Libraries


In [2]:
!pip install pytube


[notice] A new release of pip is available: 23.1.2 -> 24.0
[notice] To update, run: C:\Users\ASUS\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [ ]:
import tensorflow as tf
import keras
import cv2
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import numpy as np
import pandas as pd
from pytube import YouTube
from tensorflow.keras.models import Sequential
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Bidirectional, ConvLSTM3D,AveragePooling3D, MaxPooling3D,Input
from tensorflow.keras.layers import Bidirectional, ConvLSTM2D,AveragePooling2D, MaxPooling2D, GlobalAveragePooling2D
from tensorflow.keras.layers import Dense, Flatten, TimeDistributed, ZeroPadding3D,Dropout,BatchNormalization, LSTM
from PIL import Image
from tensorflow.keras.optimizers import Adam,Adagrad,Adadelta,SGD
from tensorflow.keras.applications import InceptionV3, DenseNet121, VGG16
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2,l1,l1_l2
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Dropout
from tensorflow.keras.utils import to_categorical,plot_model
from tqdm import tqdm
%matplotlib inline

In [ ]:
data_dir = '/kaggle/input/sports/train'
categories = os.listdir(data_dir)
num_classes = len(categories)


In [ ]:
categories

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from tensorflow.keras.utils import to_categorical,plot_model


SEQUENCE_LENGTH = 3
DIM=(224,224)

def load_groups(input_folder):
        groups = []
        for label_folder in os.listdir(input_folder):
            label_folder_path = os.path.join(input_folder, label_folder)
            if os.path.isdir(label_folder_path):
                for video_file in os.listdir(label_folder_path):
                    video_path = os.path.join(label_folder_path, video_file)
                    groups.append([video_path, label_folder])
        return groups

def frames_extraction(video_path):
    frames_list = []
    video_reader = cv2.VideoCapture(video_path)
    video_frames_count = int(video_reader.get(cv2.CAP_PROP_FRAME_COUNT))
    skip_frames_window = max(int(video_frames_count / SEQUENCE_LENGTH), 1)

    for frame_counter in range(SEQUENCE_LENGTH):
        video_reader.set(cv2.CAP_PROP_POS_FRAMES, frame_counter * skip_frames_window)
        success, frame = video_reader.read()
        if not success:
            break
        resized_frame = cv2.resize(frame, DIM)
        grayscale_frame = cv2.cvtColor(resized_frame, cv2.COLOR_BGR2GRAY)
        normalized_frame = resized_frame / 255
        frames_list.append(normalized_frame)

    video_reader.release()
    return frames_list


def split_into_train_test(groups):
    data = []
    labels = []
    original_labels = []
    label_to_index = {}
    index_to_label = {}

    for group in tqdm(groups):
        video_file_path = group[0]  # Path to the video file
        label = group[1]
        original_labels.append(label)

        if label not in label_to_index:
            index = len(label_to_index)
            label_to_index[label] = index
            index_to_label[index] = label

        frames = frames_extraction(video_file_path)  # Extract frames from the video
        if len(frames) == SEQUENCE_LENGTH:
            data.append(frames)
            labels.append(label_to_index[label])

    num_classes = len(label_to_index)
    data=np.asarray(data)
    label = np.array(label)
    
    encoded_labels = to_categorical(labels, num_classes=num_classes)

    train_data, test_data, train_labels, test_labels = train_test_split(data, encoded_labels, test_size=0.2, random_state=19, stratify=encoded_labels)

    return train_data, test_data, train_labels, test_labels, original_labels, encoded_labels



def make_csv_file(items, labels, file_path):
    items_reshaped = [np.ravel(item) for item in items]  # Reshape each array in items
    labels_reshaped = [np.ravel(label) for label in labels]  # Reshape each array in labels
    data = {'Video_Frames': items_reshaped, 'Label': labels_reshaped}
    df = pd.DataFrame(data)
    df.to_csv(file_path, index=False)


In [ ]:
groups = load_groups(data_dir)

In [ ]:
groups[0][0]

In [ ]:
train_data, test_data, train_labels, test_labels, original_labels, encoded_labels = split_into_train_test(groups)

In [ ]:
train_data[0].shape

In [ ]:
# train_data = np.reshape(train_data, (-1, 2, 224, 224, 3))
# test_data = np.reshape(test_data, (-1, 2, 224, 224, 3))

In [ ]:
# make_csv_file(train_data, train_labels, 'final_train_data.csv')
# make_csv_file(test_data, test_labels, 'final_test_data.csv')

## Data Visualization of Dataset


In [ ]:
# train_df = pd.read_csv("/kaggle/working/final_train_data.csv")
# test_df = pd.read_csv("/kaggle/working/final_test_data.csv")

In [ ]:
# train_df.columns,test_df.columns

In [ ]:
# train_df.head

In [ ]:
# test_df.head

In [ ]:
# print(f"Dimension of Training Datset is : {train_df.shape}")
# print(f"Dimension of Training Datset is : {test_df.shape}")

In [ ]:
from collections import Counter

def create_bar_plot(labels):
    # Count the occurrences of each unique label
    label_counts = Counter(labels)
    
    # Get the unique labels and their counts
    unique_labels, counts = zip(*label_counts.items())

    # Create the bar plot
    plt.figure(figsize=(10, 6))
    plt.bar(unique_labels, counts, align='center', alpha=0.7)
    plt.xlabel('Labels')
    plt.ylabel('Count')
    plt.title('Label Distribution')
    plt.xticks(rotation=45)  # Rotate x-axis labels for better readability
    plt.tight_layout()

    plt.show()
    
create_bar_plot(original_labels)

## Making CNN+LSTM Model for Training 

##### Defining the Dimesnsion of input

In [ ]:
frames = SEQUENCE_LENGTH   
height = DIM[0]
width = DIM[1]
num_classes=15

In [ ]:
def SEQ_Model():
    model = Sequential()
    
    model.add(ConvLSTM2D(filters=32, kernel_size=(3, 3), input_shape=(frames, height, width, 3),
                         strides=(1, 1), padding='same', activation='tanh', return_sequences=True, recurrent_dropout=0.2))
    model.add(BatchNormalization(momentum=0.8))
    model.add(MaxPooling3D(pool_size=(1, 2, 2), padding='same'))
    model.add(TimeDistributed(Dropout(0.2)))
    
    model.add(ConvLSTM2D(filters=64, kernel_size=(3, 3), input_shape=(frames, height, width, 3),
                         strides=(1, 1), padding='same', activation='tanh', return_sequences=True, recurrent_dropout=0.2))
    model.add(BatchNormalization(momentum=0.8))
    model.add(MaxPooling3D(pool_size=(1, 2, 2), padding='same'))
    model.add(TimeDistributed(Dropout(0.2)))
    
    model.add(ConvLSTM2D(filters=64, kernel_size=(3, 3), strides=(1, 1), padding='same', activation='tanh', return_sequences=True, recurrent_dropout=0.2))
    model.add(BatchNormalization(momentum=0.8))
    model.add(MaxPooling3D(pool_size=(1, 2, 2), padding='same'))
    model.add(TimeDistributed(Dropout(0.2)))
    
    model.add(ConvLSTM2D(filters=128, kernel_size=(3, 3), strides=(1, 1), padding='same', activation='tanh', return_sequences=True, recurrent_dropout=0.2))
    model.add(BatchNormalization(momentum=0.8))
    model.add(MaxPooling3D(pool_size=(1, 2, 2), padding='same'))
    model.add(TimeDistributed(Dropout(0.2)))
    
    model.add(Flatten())  
    

    model.add(Dense(1024, activation='relu', kernel_regularizer=l2(l2=0.01)))
    model.add(Dropout(0.3))
    model.add(Dense(2024, activation='relu', kernel_regularizer=l2(l2=0.01)))
    model.add(Dropout(0.3))
    model.add(Dense(num_classes, activation='softmax'))
    
    model.summary()
    
    return model
    

In [ ]:
def cnn_lstm_model():
    base_architecture = InceptionV3(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    base_architecture = Model(inputs=base_architecture.input, outputs=base_architecture.get_layer('mixed7').output)

    for layer in base_architecture.layers:
        layer.trainable = False

    model = Sequential()
    model.add(TimeDistributed(base_architecture, input_shape=(SEQUENCE_LENGTH, 224, 224, 3)))
    model.add(TimeDistributed(Flatten()))
    model.add(LSTM(units=64, return_sequences=True))
    model.add(LSTM(units=32, return_sequences=True))
    model.add(Flatten())
    model.add(Dense(1024, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(2048, activation='relu'))
    model.add(Dropout(0.3))
    model.add(Dense(num_classes, activation='softmax'))

    model.summary()

    return model

In [ ]:
print("Model Architecture : ")
final_model= cnn_lstm_model()
# final_model= SEQ_Model()

## Structure of Model Architecture

In [ ]:
model_str = plot_model(final_model, to_file='model.png', show_shapes=True, show_layer_names=True)
model_str

## Compile and Train the Model

In [ ]:
early_stopper = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=40,
    mode='auto',
    restore_best_weights=True,
)

# param_dist = {
#     'learning_rate': [0.001, 0.01, 0.1],
#     'momentum': [0.9, 0.95, 0.99],
#     'nesterov': [True, False],
#     'weight_decay': [0.01, 0.025, 0.05],
#     'use_ema': [True, False],
#     'ema_overwrite_frequency': [3, 5, 10]
# }

optimizer = tf.keras.optimizers.SGD(learning_rate=0.002,momentum=0.9,nesterov=True,weight_decay=0.025,use_ema=True,ema_overwrite_frequency=5)

final_model.compile(optimizer=optimizer,  
              loss='categorical_crossentropy', 
              metrics=['accuracy'])  

model_history = final_model.fit(
    train_data,
    train_labels,
    batch_size=25,
    epochs=400,
    verbose='auto',
    validation_split = 0.2,
    shuffle=True,
    use_multiprocessing=True
    ,callbacks=early_stopper
)


## Model Evaluation


In [ ]:
test_loss,test_accuracy = final_model.evaluate(test_data,test_labels)

In [ ]:
plt.figure(figsize=(12,7))
plt.suptitle("Model performance")

plt.subplot(1, 2, 1)
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title('Training and Validation Loss')
plt.plot(model_history.history["loss"], label="training loss")
plt.plot(model_history.history["val_loss"], label="validation loss")
plt.legend()


In [ ]:
plt.subplot(1, 2, 2)
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.title('Training and Validation Accuracy')
plt.plot(model_history.history['accuracy'], label="training accuracy")
plt.plot(model_history.history['val_accuracy'], label="validation accuracy")
plt.legend()
plt.show()

In [ ]:
SGD_HAR3 = final_model.save("IV3_LSTM4.h5")
SGD_HAR_wt3 = final_model.save_weights("IV3_LSTM4_wt.h5")

In [ ]:
class_labels = ['VolleyballSpiking',
 'Biking',
 'HorseRiding',
 'Fencing',
 'Archery',
 'Boxing',
 'TableTennisShot',
 'GolfSwing',
 'SoccerPenalty',
 'TennisSwing',
 'Basketball',
 'Cricket',
 'Rafting',
 'Diving',
 'JavelinThrow']

def download_youtube_frames(youtube_url):

    yt = YouTube(youtube_url)
    stream = yt.streams.get_highest_resolution()
    video_path = os.path.join("/kaggle/working", 'video.mp4')
    stream.download(output_path="/kaggle/working", filename='video')
    return video_path

def predict_single_action(input_video_file_path, SEQUENCE_LENGTH):
    frames = frames_extraction(input_video_file_path)
    plt.figure(figsize=(15, 3))
    for i in range(SEQUENCE_LENGTH):
        plt.subplot(1, 5, i + 1)
        plt.imshow(frames[i])
        plt.axis('off')
    plt.show()
    frames = np.asarray(frames)
    frames = frames.reshape(-1, SEQUENCE_LENGTH, DIM[0], DIM[1], 3)
    
    prediction = final_model.predict(frames)
    print(prediction)
    predicted_label = np.argmax(prediction, axis=1)
    
    print("Predicted label: ", predicted_label)
    print("Predicted Action: ", class_labels[predicted_label[0]])
    
    return predicted_label[0]


In [ ]:
# HourseRace 
video_link = 'https://www.youtube.com/shorts/w5oVC6vSsJE'
video_path = download_youtube_frames(video_link)
input_video_file_path = '/kaggle/working/video'
final_output = predict_single_action(input_video_file_path, SEQUENCE_LENGTH)

In [ ]:
# 'VolleyballSpiking'
video_link = 'https://www.youtube.com/shorts/Uw1XX3nfYNA'
video_path = download_youtube_frames(video_link)
input_video_file_path = '/kaggle/working/video'
final_output = predict_single_action(input_video_file_path, SEQUENCE_LENGTH)

In [ ]:
#  'Biking'
video_link = 'https://www.youtube.com/shorts/lWUmhZMr0UU'
video_path = download_youtube_frames(video_link)
input_video_file_path = '/kaggle/working/video'
final_output = predict_single_action(input_video_file_path, SEQUENCE_LENGTH)

In [ ]:
#  'BaseballPitch'
video_link = 'https://www.youtube.com/shorts/LNW4RLLrA3s'
video_path = download_youtube_frames(video_link)
input_video_file_path = '/kaggle/working/video'
final_output = predict_single_action(input_video_file_path, SEQUENCE_LENGTH)

In [ ]:
#  'SkateBoarding' --------------WRONG PREDICTION--------------------------------------------------------------------------------------
video_link = 'https://www.youtube.com/shorts/Pz_Wh0y8fYs'
video_path = download_youtube_frames(video_link)
input_video_file_path = '/kaggle/working/video'
final_output = predict_single_action(input_video_file_path, SEQUENCE_LENGTH)

In [ ]:
#  'Fencing'
video_link = 'https://www.youtube.com/shorts/54T87maKCjM'
video_path = download_youtube_frames(video_link)
input_video_file_path = '/kaggle/working/video'
final_output = predict_single_action(input_video_file_path, SEQUENCE_LENGTH)

In [ ]:
#  'RockClimbingIndoor'
video_link = 'https://www.youtube.com/shorts/SM-0MWIrsa8'
video_path = download_youtube_frames(video_link)
input_video_file_path = '/kaggle/working/video'
final_output = predict_single_action(input_video_file_path, SEQUENCE_LENGTH)

In [ ]:
#  'Rowing'
video_link = 'https://www.youtube.com/shorts/mXMgRVOpABI'
video_path = download_youtube_frames(video_link)
input_video_file_path = '/kaggle/working/video'
final_output = predict_single_action(input_video_file_path, SEQUENCE_LENGTH)

In [ ]:
#  'GolfSwin'
video_link = 'https://www.youtube.com/shorts/3vacdk_UND0'
video_path = download_youtube_frames(video_link)
input_video_file_path = '/kaggle/working/video'
final_output = predict_single_action(input_video_file_path, SEQUENCE_LENGTH)

In [ ]:
#  'TennisSwing'
video_link = 'https://www.youtube.com/shorts/SD3RSxcu9mQ'
video_path = download_youtube_frames(video_link)
input_video_file_path = '/kaggle/working/video'
final_output = predict_single_action(input_video_file_path, SEQUENCE_LENGTH)

In [ ]:
#  'Basketball' --------------WRONG PREDICTION----------------------------------------------------------------------------------------
video_link = 'https://www.youtube.com/shorts/vgVPRHVZ-M8'
video_path = download_youtube_frames(video_link)
input_video_file_path = '/kaggle/working/video'
final_output = predict_single_action(input_video_file_path, SEQUENCE_LENGTH)

In [ ]:
#  'Skiing'
video_link = 'https://www.youtube.com/shorts/_EfB_YixkEc'
video_path = download_youtube_frames(video_link)
input_video_file_path = '/kaggle/working/video'
final_output = predict_single_action(input_video_file_path, SEQUENCE_LENGTH)

In [ ]:
#  'Diving' -----------------------WRONG PREDICTION-------------------------------------------------------------------------------------
video_link = 'https://www.youtube.com/shorts/ryOBps5uEGo'
video_path = download_youtube_frames(video_link)
input_video_file_path = '/kaggle/working/video'
final_output = predict_single_action(input_video_file_path, SEQUENCE_LENGTH)

In [ ]:
#  'SoccerJuggling'
video_link = 'https://www.youtube.com/shorts/06oWyIjQpUY'
video_path = download_youtube_frames(video_link)
input_video_file_path = '/kaggle/working/video'
final_output = predict_single_action(input_video_file_path, SEQUENCE_LENGTH)

In [ ]:
#  'JavelinThrow'
video_link = 'https://www.youtube.com/shorts/uMDp3TPg-OY'
video_path = download_youtube_frames(video_link)
input_video_file_path = '/kaggle/working/video'
final_output = predict_single_action(input_video_file_path, SEQUENCE_LENGTH)

In [ ]:
# confusion matrix in range 0 to 1 and us original labels at graph at display martix
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

class_labels = ['VolleyballSpiking',
 'Biking',
 'HorseRiding',
 'Fencing',
 'Archery',
 'Boxing',
 'TableTennisShot',
 'GolfSwing',
 'SoccerPenalty',
 'TennisSwing',
 'Basketball',
 'Cricket',
 'Rafting',
 'Diving',
 'JavelinThrow']

predicted_label = final_model.predict(test_data)
predicted_label = np.argmax(predicted_label, axis=1)

cm = confusion_matrix(test_labels.argmax(axis=1), predicted_label)
print(cm)

plt.figure(figsize=(10, 10))
sns.heatmap(cm, annot=True, fmt=".3f", linewidths=.5, square=True, cmap='Blues_r',xticklabels=class_labels, yticklabels=class_labels)
plt.ylabel('Actual label')
plt.xlabel('Predicted label')
all_sample_title = 'Accuracy Score: {0}'.format(test_accuracy)
plt.title(all_sample_title, size=15)
plt.show()


In [ ]:
# !zip -r file.zip /kaggle/working

In [ ]:
# from IPython.display import FileLink
# FileLink(r'file.zip')